# Tiền xử lý dữ liệu SBA 7(a) — Phase 1

Notebook này minh họa từng bước tiền xử lý **P1.1 → P1.17** trong
[`docs/etl/etl_implementation_plan.md`](../docs/etl/etl_implementation_plan.md) §3.

- Mọi xử lý đều **gọi hàm** trong `src/etl/preprocess.py`. Notebook không chép lại logic;
  file `.py` là nguồn chuẩn, chạy độc lập bằng `python -m src.etl.preprocess`.
- **Không loại dòng nào** (BR-POP-01): không `drop_duplicates()`, không `dropna()`, không điền
  giá trị trung bình. Dòng trùng và dòng bất thường chỉ được **gắn cờ** trong `dq_issues.csv`.
  Bỏ **cột** không làm mất **dòng**.
- Các cột `Borr*` (tên/địa chỉ người vay, PII) **không được hiển thị nội dung**: chỉ hiện tên cột
  và số liệu thống kê.

In [1]:
import hashlib
import json
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'etl' / 'preprocess.py').exists())
sys.path.insert(0, str(ROOT))
from src.etl import preprocess as pp

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

PII_COLUMNS = [c for c in pp.SOURCE_COLUMNS if c.startswith('Borr')]


def an_pii(frame):
    """Thay nội dung các cột Borr* bằng '***' khi hiển thị (giữ tên cột)."""
    frame = frame.copy()
    for column in PII_COLUMNS:
        if column in frame:
            frame[column] = '***'
    return frame


def so_dong_theo_co(log):
    """Số dòng (record) theo từng DQ flag đã gắn tới thời điểm hiện tại."""
    issues = pp.issues_frame(log, pp.SOURCE.stem)
    return issues.groupby('IssueCode')['SourceRecordOrdinal'].nunique().rename('SoDong')


# Checksum TSV của lần chạy trước (lệnh python -m src.etl.preprocess) để so ở Bước 17.
manifest_path = pp.REPORT_DIR / 'manifest.json'
previous_tsv_sha256 = (json.loads(manifest_path.read_text(encoding='utf-8'))
                       ['outputs']['standardized_tsv']['sha256']
                       if manifest_path.exists() else None)
started = time.time()
print('pandas', pd.__version__, '| python', sys.version.split()[0])

pandas 2.3.3 | python 3.12.7


## Bước 1 — P1.1 Manifest: kiểm tra và đọc file nguồn

`check_source` dừng nếu file là con trỏ Git LFS, sai SHA-256 hoặc header khác 42 cột kỳ vọng.
`read_raw` đọc **mọi ô dạng chuỗi** (`dtype=str, keep_default_na=False`) để giữ số 0 đầu của
`LocationID`, `NaicsCode`, ZIP…

In [2]:
digest, header = pp.check_source(pp.SOURCE, pp.EXPECTED_SHA256)
print('File      :', pp.SOURCE.name)
print('SHA-256   :', digest)
print('Khớp kỳ vọng:', digest == pp.EXPECTED_SHA256)
print('Số cột header:', len(header))

raw = pp.read_raw(pp.SOURCE)
print('raw.shape =', raw.shape)

File      : FOIA_7a_FY2020_Present_asof_260630.csv
SHA-256   : 6c1e9132b5141a19f82bdc8ccafb86c9a01662461cad41ddb36a3cf409d8a4fe
Khớp kỳ vọng: True
Số cột header: 42


raw.shape = (388338, 42)


In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 388338 entries, 0 to 388337
Data columns (total 42 columns):
 #   Column                      Non-Null Count   Dtype 
---  ------                      --------------   ----- 
 0   AsOfDate                    388338 non-null  object
 1   Program                     388338 non-null  object
 2   LocationID                  388338 non-null  object
 3   BorrName                    388338 non-null  object
 4   BorrStreet                  388338 non-null  object
 5   BorrCity                    388338 non-null  object
 6   BorrState                   388338 non-null  object
 7   BorrZip                     388338 non-null  object
 8   BankName                    388338 non-null  object
 9   BankFDICNumber              388338 non-null  object
 10  BankNCUANumber              388338 non-null  object
 11  BankStreet                  388338 non-null  object
 12  BankCity                    388338 non-null  object
 13  BankState                   3

In [4]:
an_pii(raw).head()

,AsOfDate,Program,LocationID,BorrName,BorrStreet,BorrCity,BorrState,BorrZip,BankName,BankFDICNumber,BankNCUANumber,BankStreet,BankCity,BankState,BankZip,GrossApproval,SBAGuaranteedApproval,ApprovalDate,ApprovalFY,FirstDisbursementDate,ProcessingMethod,InitialInterestRate,FixedorVariableInterestInd,TermInMonths,NaicsCode,NaicsDescription,FranchiseCode,FranchiseName,ProjectCounty,ProjectState,SBADistrictOffice,CongressionalDistrict,BusinessType,BusinessAge,LoanStatus,PaidInFullDate,ChargeOffDate,GrossChargeOffAmount,RevolverStatus,JobsSupported,CollateralInd,SoldSecMrktInd
0,2026-06-30,7A,0029805,***,***,***,***,***,"TD Bank, National Association",18409,,2035 Limestone Rd,WILMINGTON,DE,19808,25000.0,12500.0,2019-10-01,2020,2019-10-31,SBA Express Program,11.04,V,120.0,236220,Commercial and Institutional Building Construc...,,,MONTGOMERY,PA,PHILADELPHIA DISTRICT OFFICE,04,PARTNERSHIP,Unanswered,P I F,2024-11-30,,0.0,Y,5.0,N,
1,2026-06-30,7A,0084894,***,***,***,***,***,"Santander Bank, National Association",29950,,"824 N Market St, Ste 100",WILMINGTON,DE,19801,95800.0,47900.0,2019-10-01,2020,2019-10-31,SBA Express Program,9.0,V,120.0,722511,Full-Service Restaurants,,,QUEENS,NY,NEW YORK DISTRICT OFFICE,10,CORPORATION,Existing or more than 2 years old,P I F,2023-11-30,,0.0,Y,3.0,N,
2,2026-06-30,7A,0053803,***,***,***,***,***,"U.S. Bank, National Association",06548,,425 Walnut St,Cincinnati,OH,45202,40000.0,20000.0,2019-10-01,2020,2020-02-29,SBA Express Program,6.99,F,60.0,561730,Landscaping Services,,,PLACER,CA,SACRAMENTO DISTRICT OFFICE,01,INDIVIDUAL,Unanswered,P I F,2024-11-30,,0.0,N,0.0,Y,
3,2026-06-30,7A,0123499,***,***,***,***,***,BayFirst National Bank,34997,,700 Central Avenue,Saint Petersburg,FL,33701,125000.0,106250.0,2019-10-01,2020,2019-10-31,Preferred Lenders Program,7.75,V,120.0,561990,All Other Support Services,,,GUADALUPE,TX,SAN ANTONIO DISTRICT OFFICE,15,CORPORATION,Existing or more than 2 years old,EXEMPT,,,0.0,N,3.0,N,Y
4,2026-06-30,7A,0070400,***,***,***,***,***,Golden Bank National Association,26223,,9315 Bellaire Blvd,HOUSTON,TX,77036,250000.0,187500.0,2019-10-01,2020,2019-12-31,Preferred Lenders Program,6.0,V,90.0,722511,Full-Service Restaurants,S2561,Poke Burri,HARRIS,TX,HOUSTON DISTRICT OFFICE,02,CORPORATION,Unanswered,P I F,2024-01-31,,0.0,N,4.0,Y,


**Ô trống trước xử lý.** Vì đọc với `keep_default_na=False`, `raw.isnull().sum()` luôn bằng 0:
ô trống trong CSV là chuỗi rỗng `''`, chưa phải NULL. Bảng dưới so cả hai cách đếm.

In [5]:
missing_before = pd.DataFrame({'isnull()': raw.isnull().sum(), "== ''": raw.eq('').sum()})
missing_before[missing_before["== ''"] > 0]

,isnull(),== ''
BankFDICNumber,0,41322
BankNCUANumber,0,377191
FirstDisbursementDate,0,71186
InitialInterestRate,0,8
FixedorVariableInterestInd,0,8
FranchiseCode,0,341756
FranchiseName,0,341920
CongressionalDistrict,0,24
BusinessType,0,34
BusinessAge,0,186


## Bước 2 — P1.2 SourceRecordOrdinal

Số thứ tự **record** 1..N theo thứ tự parser CSV trả về, không phải số dòng vật lý: có record chứa
xuống dòng trong ô nên file có nhiều dòng vật lý hơn số record.

In [6]:
ordinal = pp.record_ordinal(raw)
with pp.SOURCE.open('rb') as handle:
    physical_lines = sum(1 for _ in handle)
print('Ordinal từ', ordinal.min(), 'đến', ordinal.max(), '| duy nhất:', ordinal.is_unique)
print('Số record        :', len(raw))
print('Số dòng vật lý   :', physical_lines, '(gồm 1 dòng header)')

Ordinal từ 1 đến 388338 | duy nhất: True
Số record        : 388338
Số dòng vật lý   : 388355 (gồm 1 dòng header)


## Bước 3 — P1.3 Trim khoảng trắng, chuỗi rỗng → NULL

`clean_columns` áp `clean_text` cho cả 42 cột: bỏ khoảng trắng đầu/cuối, chuỗi rỗng thành NULL.
**Không đổi chữ hoa/thường.**

In [7]:
clean = pp.clean_columns(raw)
print('Program trước:', [repr(v) for v in raw['Program'].unique()])
print('Program sau  :', [repr(v) for v in clean['Program'].unique()])

Program trước: ["' 7A'"]
Program sau  : ["'7A'"]


In [8]:
missing_compare = pd.DataFrame({"Trước: == ''": raw.eq('').sum(),
                                'Sau: isnull()': clean.isnull().sum()})
missing_compare

,Trước: == '',Sau: isnull()
AsOfDate,0,0
Program,0,0
LocationID,0,0
BorrName,0,0
BorrStreet,0,0
BorrCity,0,0
BorrState,0,0
BorrZip,0,0
BankName,0,0
BankFDICNumber,41322,41322


Số giá trị khác nhau trước/sau trim của các cột phân tích (NULL tính là một giá trị):

In [9]:
pd.DataFrame(pp.trim_cardinality(raw, clean)).T.loc[pp.ANALYSIS_COLUMNS]

,raw,standardized
ProjectState,54,54
ProjectCounty,1855,1855
NaicsCode,1157,1157
NaicsDescription,1783,1131
LocationID,2338,2338
BankName,2183,2183
ProcessingMethod,19,19
LoanStatus,5,5
BusinessType,4,4
BusinessAge,6,6


## Bước 4 — P1.4 Ô nhiều dòng

`\r`, `\n`, `\t` trong text được thay bằng một dấu cách (đã làm trong `clean_text` ở Bước 3),
để file đầu ra có đúng 1 record trên 1 dòng.

In [10]:
print('Record có xuống dòng trong ô (raw):', pp.multiline_records(raw))
left = clean.apply(lambda s: s.str.contains('[\r\n\t]', regex=True, na=False)).sum().sum()
print('Ô còn ký tự \\r \\n \\t sau xử lý :', int(left))

Record có xuống dòng trong ô (raw): 16


Ô còn ký tự \r \n \t sau xử lý : 0


## Bước 5 — P1.5 Parse kiểu dữ liệu

- 5 cột ngày → ISO `YYYY-MM-DD`; `GrossApproval`, `SBAGuaranteedApproval`, `GrossChargeOffAmount`
  → số thập phân, **giữ nguyên độ chính xác gốc** (D4: DECIMAL(18,3));
  `TermInMonths`, `JobsSupported`, `ApprovalFY` → số nguyên.
- Cột mã (`LocationID`, `NaicsCode`, ZIP, FDIC/NCUA, district) **giữ dạng text**.
- Parse lỗi → cờ `PARSE_ERROR`; không sửa raw. Mọi cờ DQ được gom vào `IssueLog`.

In [11]:
log = pp.IssueLog()
typed = pp.parse_types(clean, ordinal, log)
formatted = pp.format_typed(clean, typed, ordinal, log)
formatted.head()

,ApprovalFY,TermInMonths,JobsSupported,GrossApproval,SBAGuaranteedApproval,GrossChargeOffAmount
0,2020,120,5,25000.00,12500.00,0.00
1,2020,120,3,95800.00,47900.00,0.00
2,2020,60,0,40000.00,20000.00,0.00
3,2020,120,3,125000.00,106250.00,0.00
4,2020,90,4,250000.00,187500.00,0.00


In [12]:
issues = pp.issues_frame(log, pp.SOURCE.stem)
print('Số dòng theo cờ sau Bước 5:')
print(so_dong_theo_co(log).to_string())
issues[issues['IssueCode'] == 'AMOUNT_SCALE_GT_2']

Số dòng theo cờ sau Bước 5:
IssueCode
AMOUNT_SCALE_GT_2    2


,SourceFileID,SourceRecordOrdinal,IssueCode,ColumnName,RawValue,Detail
0,FOIA_7a_FY2020_Present_asof_260630,81174,AMOUNT_SCALE_GT_2,SBAGuaranteedApproval,3749992.425,"kept at source precision; DECIMAL(x,2) would r..."
1,FOIA_7a_FY2020_Present_asof_260630,84754,AMOUNT_SCALE_GT_2,SBAGuaranteedApproval,3749975.124,"kept at source precision; DECIMAL(x,2) would r..."


## Bước 6 — P1.6 Ngày và năm tài chính

`FiscalYearDerived` suy từ `ApprovalDate` (tháng ≥ 10 → năm + 1) và so với `ApprovalFY` của nguồn.
Gắn cờ `ChargeOffDate > AsOfDate`, `PaidInFullDate < ApprovalDate`, `CHGOFF` thiếu `ChargeOffDate`.
Chỉ gắn cờ, không sửa/xóa.

In [13]:
fiscal = pp.check_dates(clean, typed, formatted, ordinal, log)
pd.crosstab(formatted['ApprovalFY'], fiscal, rownames=['ApprovalFY'], colnames=['FiscalYearDerived'])

FiscalYearDerived,2020,2021,2022,2023,2024,2025,2026
ApprovalFY,,,,,,,
2020,42298,0,0,0,0,0,0
2021,0,51856,0,0,0,0,0
2022,0,0,47678,0,0,0,0
2023,0,0,0,57362,0,0,0
2024,0,0,0,0,70242,0,0
2025,0,0,0,0,0,78078,0
2026,0,0,0,0,0,0,40824


In [14]:
so_dong_theo_co(log)

IssueCode
AMOUNT_SCALE_GT_2        2
CHARGEOFF_AFTER_ASOF    22
CHGOFF_MISSING_DATE      5
PIF_BEFORE_APPROVAL      2
Name: SoDong, dtype: int64

## Bước 7 — P1.7 Kiểm tra số

Gắn cờ amount âm, `SBAGuaranteedApproval > GrossApproval`, `TermInMonths = 0`,
`InitialInterestRate = 0`. Giá trị 0 được giữ, khác NULL.

In [15]:
pp.check_numbers(clean, typed, ordinal, log)
so_dong_theo_co(log)

IssueCode
AMOUNT_SCALE_GT_2        2
CHARGEOFF_AFTER_ASOF    22
CHGOFF_MISSING_DATE      5
PIF_BEFORE_APPROVAL      2
RATE_ZERO               87
TERM_ZERO               11
Name: SoDong, dtype: int64

## Bước 8 — P1.8 Dòng trùng tuyệt đối

`RowHash` (SHA-256) trên 42 cột raw → `DuplicateGroupID`, `IsExactDuplicate`.
**Chỉ gắn cờ, không xóa** — SBA không công bố LoanID nên không chứng minh được là cùng khoản vay.

In [16]:
dups = pp.flag_duplicates(raw, ordinal, log)
print('Dòng thuộc nhóm trùng:', int(dups['IsExactDuplicate'].sum()))
print('Số nhóm trùng        :', dups['DuplicateGroupID'].nunique())
print('Số dòng sau bước này :', len(raw), '(không đổi)')
dups[dups['IsExactDuplicate']].head()

Dòng thuộc nhóm trùng: 687
Số nhóm trùng        : 296
Số dòng sau bước này : 388338 (không đổi)


,RowHash,GroupSize,DuplicateGroupID,IsExactDuplicate
39,48c33e2a7f7a31ca522ec68c1af1106699aae4a70fe9c1...,2,1,True
71,8e5d3e3940ffe8034fb977f8dde7440337d7c3dcd9620f...,4,2,True
103,8e5d3e3940ffe8034fb977f8dde7440337d7c3dcd9620f...,4,2,True
113,8e5d3e3940ffe8034fb977f8dde7440337d7c3dcd9620f...,4,2,True
119,8e5d3e3940ffe8034fb977f8dde7440337d7c3dcd9620f...,4,2,True


## Bước 9 — P1.9 Trạng thái khoản vay

`RawStatus` giữ đúng nguồn; `CanonicalStatus`: `P I F` → `PIF`, giá trị khác giữ nguyên,
giá trị lạ → `UNMAPPED` (BR-STATUS-01).

In [17]:
status = pp.map_status(clean)
pd.crosstab(status['RawStatus'], status['CanonicalStatus'])

CanonicalStatus,CANCLD,CHGOFF,COMMIT,EXEMPT,PIF
RawStatus,,,,,
CANCLD,50104,0,0,0,0
CHGOFF,0,6893,0,0,0
COMMIT,0,0,21079,0,0
EXEMPT,0,0,0,242061,0
P I F,0,0,0,0,68201


## Bước 10 — P1.10 TermBand

`ZERO`=0, `SHORT`=1–60, `MEDIUM`=61–119, `TERM_120`=120, `LONG`=121–240, `VERY_LONG`>240,
`MISSING`, `INVALID` (BR-TERM-01).

In [18]:
band_order = ['ZERO', 'SHORT', 'MEDIUM', 'TERM_120', 'LONG', 'VERY_LONG', 'MISSING', 'INVALID']
band = pp.map_term_band(clean)
band_counts = band.value_counts().reindex(band_order, fill_value=0).rename('SoDong')
print('Tổng các band:', int(band_counts.sum()))
band_counts

Tổng các band: 388338


TermInMonths
ZERO             11
SHORT         32801
MEDIUM        39550
TERM_120     236672
LONG          26530
VERY_LONG     52774
MISSING           0
INVALID           0
Name: SoDong, dtype: int64

## Bước 11 — P1.11 NAICS Sector (D1 = A)

`NaicsSectorCode` = 2 số đầu, gộp 31–33 / 44–45 / 48–49. Chưa xác minh với bảng NAICS chính thức nên
`SectorMappingStatus = CANDIDATE_UNVERIFIED`; mã không đủ 6 chữ số → `UNMAPPED`.

In [19]:
naics = pp.map_naics(clean)
naics['SectorMappingStatus'].value_counts(dropna=False).rename('SoDong')

SectorMappingStatus
CANDIDATE_UNVERIFIED    388338
Name: SoDong, dtype: int64

In [20]:
naics['NaicsSectorCode'].value_counts(dropna=False).sort_index().rename('SoDong')

NaicsSectorCode
11        4049
21         682
22         524
23       49717
31-33    24111
42       16676
44-45    47279
48-49    22275
51        3717
52        7178
53        9360
54       36973
55         241
56       22291
61        5959
62       37073
71       13825
72       49021
81       37272
92         115
Name: SoDong, dtype: int64

## Bước 12 — P1.12 BusinessType × BusinessAge

Giữ nhãn gốc; `*ValueStatus` = `PRESENT`/`MISSING`. `Unanswered` là giá trị PRESENT, khác ô trống.

In [21]:
business = pp.map_business(clean)
combos = business[['BusinessTypeRaw', 'BusinessAgeRaw']].drop_duplicates()
print('Số tổ hợp BusinessType × BusinessAge:', len(combos))
pd.crosstab(business['BusinessTypeRaw'].fillna('(trống)'),
            business['BusinessAgeRaw'].fillna('(trống)'))

Số tổ hợp BusinessType × BusinessAge: 22


BusinessAgeRaw,(trống),Change of Ownership,Existing or more than 2 years old,New Business or 2 years or less,"Startup, Loan Funds will Open Business",Unanswered
BusinessTypeRaw,,,,,,
(trống),0,7,11,8,8,0
CORPORATION,174,39344,199648,60866,61292,3885
INDIVIDUAL,10,819,14015,2520,1178,342
PARTNERSHIP,2,385,2874,488,411,51


## Bước 13 — P1.13 Cột lookup cho SSIS

`assemble_standardized` ghép các cột ở Bước 2–12 thành bảng TSV và thêm cột `*Lookup`: NULL →
`(Missing)` để SSIS Lookup không phải so NULL = NULL. Cột gốc vẫn giữ NULL.

In [22]:
standardized = pp.assemble_standardized(pp.SOURCE.stem, ordinal, clean, typed, formatted,
                                        fiscal, dups, log)
cleaned = pp.assemble_clean_copy(clean, typed, formatted)
print('standardized.shape =', standardized.shape)
(standardized[list(pp.LOOKUP_COLUMNS.values())] == pp.MISSING_TOKEN).sum().rename('So_(Missing)')

standardized.shape = (388338, 38)


ProjectStateLookup          0
ProjectCountyLookup         0
NaicsCodeLookup             0
NaicsDescriptionLookup      0
LocationIDLookup            0
ProcessingMethodLookup      0
RawStatusLookup             0
BusinessTypeLookup         34
BusinessAgeLookup         186
Name: So_(Missing), dtype: int64

**Tổng hợp DQ flag** (số dòng theo từng cờ; một dòng có thể mang nhiều cờ):

In [23]:
so_dong_theo_co(log)

IssueCode
AMOUNT_SCALE_GT_2         2
CHARGEOFF_AFTER_ASOF     22
CHGOFF_MISSING_DATE       5
EXACT_DUPLICATE         687
PIF_BEFORE_APPROVAL       2
RATE_ZERO                87
TERM_ZERO                11
Name: SoDong, dtype: int64

## Bước 14 — P1.14 Xuất file và đối soát

`export_outputs` ghi TSV cho SSIS, bản sạch `_clean.csv` của file gốc, `dq_issues.csv`,
`reconciliation.csv`, `column_selection.csv`, `before_after.csv`, `manifest.json`.
Đối soát đọc lại file từ đĩa và so với raw.

In [24]:
audit = pp.audit_summary(raw, clean, standardized, dups)
manifest = pp.export_outputs(raw, standardized, cleaned, log, audit, source=pp.SOURCE,
                             digest=digest, header=header, started=started)
recon = pd.read_csv(pp.REPORT_DIR / 'reconciliation.csv', dtype=str, keep_default_na=False)
print('Đối soát khớp toàn bộ:', manifest['outputs']['reconciliation_all_match'])
recon

Đối soát khớp toàn bộ: True


,Output,Metric,Key,RawValue,OutputValue,Match
0,standardized_tsv,row_count,,388338,388338,True
1,standardized_tsv,sum_GrossApproval,,202550139718.0,202550139718.00,True
2,standardized_tsv,sum_SBAGuaranteedApproval,,152596337956.289,152596337956.289,True
3,standardized_tsv,sum_GrossChargeOffAmount,,898807495.32,898807495.32,True
4,standardized_tsv,sum_JobsSupported,,4072432.0,4072432,True
5,standardized_tsv,count_by_ApprovalFY,2020,42298,42298,True
6,standardized_tsv,count_by_ApprovalFY,2021,51856,51856,True
7,standardized_tsv,count_by_ApprovalFY,2022,47678,47678,True
8,standardized_tsv,count_by_ApprovalFY,2023,57362,57362,True
9,standardized_tsv,count_by_ApprovalFY,2024,70242,70242,True


In [25]:
tsv_path = pp.STAGING_DIR / pp.TSV_NAME
tsv = pp.read_tsv(tsv_path)
print('tsv.shape =', tsv.shape)
tsv.info()

tsv.shape = (388338, 38)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 388338 entries, 0 to 388337
Data columns (total 38 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   SourceFileID             388338 non-null  object
 1   SourceRecordOrdinal      388338 non-null  object
 2   AsOfDate                 388338 non-null  object
 3   ApprovalDate             388338 non-null  object
 4   ApprovalFY               388338 non-null  object
 5   FiscalYearDerived        388338 non-null  object
 6   ProjectState             388338 non-null  object
 7   ProjectCounty            388338 non-null  object
 8   NaicsCode                388338 non-null  object
 9   NaicsDescription         388338 non-null  object
 10  NaicsSectorCode          388338 non-null  object
 11  SectorMappingStatus      388338 non-null  object
 12  LocationID               388338 non-null  object
 13  BankName                 388338 non-null  object


In [26]:
tsv.head()

,SourceFileID,SourceRecordOrdinal,AsOfDate,ApprovalDate,ApprovalFY,FiscalYearDerived,ProjectState,ProjectCounty,NaicsCode,NaicsDescription,NaicsSectorCode,SectorMappingStatus,LocationID,BankName,ProcessingMethod,RawStatus,CanonicalStatus,TermInMonths,TermBandCode,BusinessTypeRaw,BusinessAgeRaw,BusinessTypeValueStatus,BusinessAgeValueStatus,GrossApproval,SBAGuaranteedApproval,GrossChargeOffAmount,JobsSupported,IsExactDuplicate,DQFlagCount,ProjectStateLookup,ProjectCountyLookup,NaicsCodeLookup,NaicsDescriptionLookup,LocationIDLookup,ProcessingMethodLookup,RawStatusLookup,BusinessTypeLookup,BusinessAgeLookup
0,FOIA_7a_FY2020_Present_asof_260630,1,2026-06-30,2019-10-01,2020,2020,PA,MONTGOMERY,236220,Commercial and Institutional Building Construc...,23,CANDIDATE_UNVERIFIED,0029805,"TD Bank, National Association",SBA Express Program,P I F,PIF,120,TERM_120,PARTNERSHIP,Unanswered,PRESENT,PRESENT,25000.00,12500.00,0.00,5,0,0,PA,MONTGOMERY,236220,Commercial and Institutional Building Construc...,0029805,SBA Express Program,P I F,PARTNERSHIP,Unanswered
1,FOIA_7a_FY2020_Present_asof_260630,2,2026-06-30,2019-10-01,2020,2020,NY,QUEENS,722511,Full-Service Restaurants,72,CANDIDATE_UNVERIFIED,0084894,"Santander Bank, National Association",SBA Express Program,P I F,PIF,120,TERM_120,CORPORATION,Existing or more than 2 years old,PRESENT,PRESENT,95800.00,47900.00,0.00,3,0,0,NY,QUEENS,722511,Full-Service Restaurants,0084894,SBA Express Program,P I F,CORPORATION,Existing or more than 2 years old
2,FOIA_7a_FY2020_Present_asof_260630,3,2026-06-30,2019-10-01,2020,2020,CA,PLACER,561730,Landscaping Services,56,CANDIDATE_UNVERIFIED,0053803,"U.S. Bank, National Association",SBA Express Program,P I F,PIF,60,SHORT,INDIVIDUAL,Unanswered,PRESENT,PRESENT,40000.00,20000.00,0.00,0,0,0,CA,PLACER,561730,Landscaping Services,0053803,SBA Express Program,P I F,INDIVIDUAL,Unanswered
3,FOIA_7a_FY2020_Present_asof_260630,4,2026-06-30,2019-10-01,2020,2020,TX,GUADALUPE,561990,All Other Support Services,56,CANDIDATE_UNVERIFIED,0123499,BayFirst National Bank,Preferred Lenders Program,EXEMPT,EXEMPT,120,TERM_120,CORPORATION,Existing or more than 2 years old,PRESENT,PRESENT,125000.00,106250.00,0.00,3,0,0,TX,GUADALUPE,561990,All Other Support Services,0123499,Preferred Lenders Program,EXEMPT,CORPORATION,Existing or more than 2 years old
4,FOIA_7a_FY2020_Present_asof_260630,5,2026-06-30,2019-10-01,2020,2020,TX,HARRIS,722511,Full-Service Restaurants,72,CANDIDATE_UNVERIFIED,0070400,Golden Bank National Association,Preferred Lenders Program,P I F,PIF,90,MEDIUM,CORPORATION,Unanswered,PRESENT,PRESENT,250000.00,187500.00,0.00,4,0,0,TX,HARRIS,722511,Full-Service Restaurants,0070400,Preferred Lenders Program,P I F,CORPORATION,Unanswered


Ô NULL sau xử lý (bảng chuẩn hóa trong bộ nhớ, NULL = `None`; trong file TSV là ô trống):

In [27]:
standardized.isnull().sum()

SourceFileID                 0
SourceRecordOrdinal          0
AsOfDate                     0
ApprovalDate                 0
ApprovalFY                   0
FiscalYearDerived            0
ProjectState                 0
ProjectCounty                0
NaicsCode                    0
NaicsDescription             0
NaicsSectorCode              0
SectorMappingStatus          0
LocationID                   0
BankName                     0
ProcessingMethod             0
RawStatus                    0
CanonicalStatus              0
TermInMonths                 0
TermBandCode                 0
BusinessTypeRaw             34
BusinessAgeRaw             186
BusinessTypeValueStatus      0
BusinessAgeValueStatus       0
GrossApproval                0
SBAGuaranteedApproval        0
GrossChargeOffAmount         0
JobsSupported                0
IsExactDuplicate             0
DQFlagCount                  0
ProjectStateLookup           0
ProjectCountyLookup          0
NaicsCodeLookup              0
NaicsDes

In [28]:
pd.concat({'SectorMappingStatus': tsv['SectorMappingStatus'].value_counts(),
           'TermBandCode': tsv['TermBandCode'].value_counts()}).rename('SoDong')

SectorMappingStatus  CANDIDATE_UNVERIFIED    388338
TermBandCode         TERM_120                236672
                     VERY_LONG                52774
                     MEDIUM                   39550
                     SHORT                    32801
                     LONG                     26530
                     ZERO                        11
Name: SoDong, dtype: int64

## Bước 15 — P1.15 Bảng chọn cột

Đủ 42 cột nguồn. `Decision` suy ra từ danh sách cột TSV thực tế; `MissingCount`/`DistinctCount`
tính từ dữ liệu (trống sau trim = thiếu). Mã lý do: `PII`, `NGOAI_PHAM_VI_Q1_Q15`,
`THIEU_TREN_50PCT`, `HANG_SO`, `CHI_DUNG_DQ`, `CO_TRONG_KHO`.

In [29]:
selection = pd.read_csv(pp.REPORT_DIR / 'column_selection.csv', dtype=str, keep_default_na=False)
print('Số cột KEEP/DROP:', selection['Decision'].value_counts().to_dict())
selection

Số cột KEEP/DROP: {'DROP': 24, 'KEEP': 18}


,SourceColumn,MissingCount,MissingPct,DistinctCount,Decision,TargetColumns,Reason
0,AsOfDate,0,0.0,1,KEEP,AsOfDate,CO_TRONG_KHO
1,Program,0,0.0,1,DROP,,HANG_SO
2,LocationID,0,0.0,2338,KEEP,LocationID|LocationIDLookup,CO_TRONG_KHO
3,BorrName,0,0.0,323606,DROP,,PII
4,BorrStreet,0,0.0,333420,DROP,,PII
5,BorrCity,0,0.0,21011,DROP,,PII
6,BorrState,0,0.0,54,DROP,,PII
7,BorrZip,0,0.0,21250,DROP,,PII
8,BankName,0,0.0,2183,KEEP,BankName,CO_TRONG_KHO
9,BankFDICNumber,41322,10.64,1873,DROP,,NGOAI_PHAM_VI_Q1_Q15


## Bước 16 — P1.16 Bảng trước/sau

Số dòng giữ nguyên 388.338; số cột giảm vì bỏ cột không dùng; tổng tiền không đổi.

In [30]:
pd.read_csv(pp.REPORT_DIR / 'before_after.csv', dtype=str, keep_default_na=False)

,Metric,Column,Before,After
0,row_count,,388338,388338
1,column_count,,42,38
2,blank_cells,AsOfDate,0,0
3,blank_cells,LocationID,0,0
4,blank_cells,BankName,0,0
5,blank_cells,GrossApproval,0,0
6,blank_cells,SBAGuaranteedApproval,0,0
7,blank_cells,ApprovalDate,0,0
8,blank_cells,ApprovalFY,0,0
9,blank_cells,ProcessingMethod,0,0


## Bước 17 — P1.17 Kiểm tra notebook cho cùng kết quả với script

TSV do notebook tạo phải có cùng SHA-256 với lần chạy `python -m src.etl.preprocess` trước đó.

In [31]:
notebook_sha256 = hashlib.sha256(tsv_path.read_bytes()).hexdigest()
print('SHA-256 TSV (notebook)    :', notebook_sha256)
print('SHA-256 TSV (lần chạy trước):', previous_tsv_sha256)
print('Giống nhau:', notebook_sha256 == previous_tsv_sha256)
print('Số dòng: raw', len(raw), '→ TSV', len(tsv))

SHA-256 TSV (notebook)    : 837723fb7b6b8358ce4aaf4855ea396b197104660babbe6d6d9f6a6be3a78bc5
SHA-256 TSV (lần chạy trước): 837723fb7b6b8358ce4aaf4855ea396b197104660babbe6d6d9f6a6be3a78bc5
Giống nhau: True
Số dòng: raw 388338 → TSV 388338
